# 01 · Build the harmonised dataset (Colab)

Builds the leakage-safe Govi Isuru dataset on Google Drive:
fetch → label harmonisation → normalise → near-duplicate grouping → splits → `manifest.csv` + `build_report.json`.

**Before you start**
1. Runtime → *Change runtime type* → CPU is enough for this notebook.
2. Manual sources (Mendeley): download and unzip into `DATA_ROOT/raw/<source_id>/` — e.g. CS-D from https://data.mendeley.com/datasets/94fzcdz8gz/1 → `raw/csd_assam/`.
3. Legacy v1 data (optional, not used for paper numbers): copy `tea_dataset/` and `chili_dataset/` to `DATA_ROOT/legacy/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/govi-ai/data'
!mkdir -p $DATA_ROOT

In [ ]:
!git clone -q https://github.com/Kalana-Lakshan/govi-isuru-ai-service.git /content/govi-isuru-ai-service || git -C /content/govi-isuru-ai-service pull -q
%cd /content/govi-isuru-ai-service
!pip install -q -e ".[data]"
!python scripts/build_dataset.py --list

## Smoke test (≈1 min)
Builds 50 images from two small sources into a scratch folder, to check everything works.

In [ ]:
!python scripts/build_dataset.py --data-root /content/smoke --sources uci_rice tea_sickness --limit 50

## Full build
Re-runnable: sources that already have `manifests/<id>.csv` are skipped (use `--force` to rebuild one).
Manual sources that are not downloaded yet are reported and skipped. Expect roughly 1–2 h and ~6–10 GB on Drive.

In [ ]:
!python scripts/build_dataset.py --data-root $DATA_ROOT

In [ ]:
import json, pandas as pd
report = json.load(open(f'{DATA_ROOT}/build_report.json'))
print(json.dumps({k: report[k] for k in ['rows', 'by_role', 'by_split', 'dedupe', 'known_classes_without_images']}, indent=1))
m = pd.read_csv(f'{DATA_ROOT}/manifest.csv')
m[m.role == 'known'].pivot_table(index=['crop', 'label'], columns='source', values='sample_id', aggfunc='count', fill_value=0)

## Legacy baseline (the paper's "before" numbers)
Copy the v1 `ai-service/models/` folder to `/content/drive/MyDrive/govi-ai/legacy_models/` first.

In [ ]:
!pip install -q -e ".[legacy]"
!python scripts/eval_legacy.py --data-root $DATA_ROOT --legacy-models /content/drive/MyDrive/govi-ai/legacy_models --out /content/drive/MyDrive/govi-ai/artifacts/legacy_baseline.json